In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## Import

In [ ]:
import torch

# Check if CUDA is available
print("CUDA Available:", torch.cuda.is_available())

CUDA Available: True


In [ ]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/SteamBoat/examples")
cwd = os.getcwd()
print(cwd)

sys.path.append("../")
if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/SteamBoat/examples
GPU:  Tesla T4


In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
from tqdm.notebook import tqdm
import scipy as sp
import numpy as np
import multiprocessing
import pickle as pkl
import gc
import sklearn.metrics

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [ ]:
import spatialdata as sd
import steamboat as sf
import steamboat.model as sm
#importlib.reload(sm)
#import steamboat.integrated_model
# importlib.reload(spaceformer.benchmarks)

In [ ]:
import importlib
import steamboat.tools
import steamboat.model

In [ ]:
importlib.reload(sf)
importlib.reload(sm)

<module 'steamboat.model' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/SteamBoat/examples/../steamboat/model.py'>

## Train

In [ ]:
adata = sc.read_h5ad("../../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

In [ ]:
adatas = []
for i in adata.obs['region'].unique():
    adatas.append(adata[adata.obs['region'] == i])
    adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.

/tmp/ipykernel_6734/4759537.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.


In [ ]:
adatas = sf.prep_adatas(adatas, norm=True, log1p=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
dataset = sf.make_dataset(adatas, sparse_graph=True, regional_obs=['global'])

Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.
Using ['global'] as regional annotations.


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import gc
torch.cuda.empty_cache()

In [ ]:
sf.set_random_seed(0)
model = sm.Steamboat(adatas[0].var_names.tolist(), n_heads=64, n_scales=3)
model = model.to(device)
# model.load_state_dict(torch.load('saved_models/mmbrain_new.pth', weights_only=True))


In [ ]:
# masking_rate=0.8
model.fit(dataset, entry_masking_rate=0.8, feature_masking_rate=0,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          sched=torch.optim.lr_scheduler.OneCycleLR,
          #sched= None,
          max_lr=0.1, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=200, stop_tol=200)

[2026-05-10 19:59:49,559::train::INFO] Epoch 1: train_loss 0.10503
INFO:train:Epoch 1: train_loss 0.10503
[2026-05-10 20:01:32,407::train::INFO] Epoch 201: train_loss 0.09143
INFO:train:Epoch 201: train_loss 0.09143
[2026-05-10 20:03:15,860::train::INFO] Epoch 401: train_loss 0.06838
INFO:train:Epoch 401: train_loss 0.06838
[2026-05-10 20:04:59,683::train::INFO] Epoch 601: train_loss 0.06098
INFO:train:Epoch 601: train_loss 0.06098
[2026-05-10 20:06:43,504::train::INFO] Epoch 801: train_loss 0.05869
INFO:train:Epoch 801: train_loss 0.05869
[2026-05-10 20:08:27,244::train::INFO] Epoch 1001: train_loss 0.05818
INFO:train:Epoch 1001: train_loss 0.05818
[2026-05-10 20:10:11,109::train::INFO] Epoch 1201: train_loss 0.05776
INFO:train:Epoch 1201: train_loss 0.05776
[2026-05-10 20:11:55,204::train::INFO] Epoch 1401: train_loss 0.05730
INFO:train:Epoch 1401: train_loss 0.05730
[2026-05-10 20:13:39,306::train::INFO] Epoch 1601: train_loss 0.05692
INFO:train:Epoch 1601: train_loss 0.05692
[2026-

Steamboat(
  (spatial_gather): BilinearAttention(
    (bias): NonNegBias(
      (elu): ELU(alpha=1.0)
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (cosine_similarity): CosineSimilarity()
  )
)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), 'saved_models/Mouse_Brain_64_0.8.pth')